# Data Exploration

Quick exploration of CelebAMask-HQ and FFHQ datasets.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

from src.data.dataset import CelebAMaskHQDataset, FFHQDataset, CELEBA_ATTRIBUTES

## 1. CelebAMask-HQ Dataset

In [ ]:
# Load training dataset
celeba_path = '../data/CelebAMask-HQ'

if Path(celeba_path).exists():
    train_dataset = CelebAMaskHQDataset(celeba_path, split='train')
    print(f"Training samples: {len(train_dataset)}")
    
    val_dataset = CelebAMaskHQDataset(celeba_path, split='val')
    print(f"Validation samples: {len(val_dataset)}")
    
    test_dataset = CelebAMaskHQDataset(celeba_path, split='test')
    print(f"Test samples: {len(test_dataset)}")
else:
    print(f"Dataset not found at {celeba_path}")

In [ ]:
# Visualize samples
if Path(celeba_path).exists():
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    
    for i, ax in enumerate(axes.flat):
        sample = train_dataset[i]
        img = sample['image'].permute(1, 2, 0).numpy()
        
        # Denormalize
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = img * std + mean
        img = np.clip(img, 0, 1)
        
        ax.imshow(img)
        ax.axis('off')
        
    plt.suptitle('CelebAMask-HQ Samples')
    plt.tight_layout()
    plt.show()

In [ ]:
# Verify attributes for a random image
import random

if Path(celeba_path).exists():
    idx = random.randint(0, len(train_dataset) - 1)
    sample = train_dataset[idx]
    
    img = sample['image'].permute(1, 2, 0).numpy()
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img * std + mean
    img = np.clip(img, 0, 1)
    
    attrs = sample['attributes'].numpy()
    
    # Show image
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(f"Image idx: {sample['idx']}")
    plt.axis('off')
    plt.show()
    
    # Show positive attributes
    print(f"\nPositive attributes for image {sample['idx']}:")
    positive_attrs = [CELEBA_ATTRIBUTES[i] for i, v in enumerate(attrs) if v > 0.5]
    print(", ".join(positive_attrs) if positive_attrs else "None")

In [ ]:
# Attribute distribution
if Path(celeba_path).exists():
    attrs = []
    for i in range(min(1000, len(train_dataset))):
        sample = train_dataset[i]
        attrs.append(sample['attributes'].numpy())
    
    attrs = np.stack(attrs)
    attr_means = attrs.mean(axis=0)
    
    plt.figure(figsize=(15, 5))
    plt.bar(range(40), attr_means)
    plt.xticks(range(40), CELEBA_ATTRIBUTES, rotation=90)
    plt.ylabel('Frequency')
    plt.title('Attribute Distribution')
    plt.tight_layout()
    plt.show()

## 2. Inpainting Masks

In [ ]:
# Load inpainting dataset
if Path(celeba_path).exists():
    inpaint_dataset = CelebAMaskHQDataset(celeba_path, split='train', for_inpainting=True)
    
    fig, axes = plt.subplots(3, 4, figsize=(12, 9))
    
    for i in range(3):
        sample = inpaint_dataset[i]
        
        # Original
        img = sample['image'].permute(1, 2, 0).numpy()
        axes[i, 0].imshow(img)
        axes[i, 0].set_title('Original')
        axes[i, 0].axis('off')
        
        # Mask
        mask = sample['mask'].squeeze().numpy()
        axes[i, 1].imshow(mask, cmap='gray')
        axes[i, 1].set_title('Mask')
        axes[i, 1].axis('off')
        
        # Masked image
        masked = sample['masked_image'].permute(1, 2, 0).numpy()
        axes[i, 2].imshow(masked)
        axes[i, 2].set_title('Masked')
        axes[i, 2].axis('off')
        
        # Overlay
        overlay = img.copy()
        overlay[mask > 0.5] = [1, 0, 0]
        axes[i, 3].imshow(overlay)
        axes[i, 3].set_title('Overlay')
        axes[i, 3].axis('off')
    
    plt.suptitle('Inpainting Masks')
    plt.tight_layout()
    plt.show()

## 3. FFHQ Gallery

In [ ]:
# Load FFHQ
ffhq_path = '../data/FFHQ'

if Path(ffhq_path).exists():
    ffhq_dataset = FFHQDataset(ffhq_path, max_images=1000)
    print(f"FFHQ samples: {len(ffhq_dataset)}")
    
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    
    for i, ax in enumerate(axes.flat):
        sample = ffhq_dataset[i]
        img = sample['image'].permute(1, 2, 0).numpy()
        
        # Denormalize
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = img * std + mean
        img = np.clip(img, 0, 1)
        
        ax.imshow(img)
        ax.axis('off')
        
    plt.suptitle('FFHQ Samples')
    plt.tight_layout()
    plt.show()
else:
    print(f"FFHQ not found at {ffhq_path}")

## 4. Model Summary

In [ ]:
import torch
from src.models.encoder import FaceEncoder
from src.models.unet import UNet

# Encoder
encoder = FaceEncoder()
total_params = sum(p.numel() for p in encoder.parameters())
print(f"Encoder parameters: {total_params:,}")

# Test forward pass
x = torch.randn(1, 3, 256, 256)
out = encoder(x)
print(f"Embedding shape: {out['embedding'].shape}")
print(f"Attributes shape: {out['attributes'].shape}")

In [ ]:
# U-Net
unet = UNet(in_channels=4, out_channels=3)
total_params = sum(p.numel() for p in unet.parameters())
print(f"U-Net parameters: {total_params:,}")

# Test forward pass
x = torch.randn(1, 4, 256, 256)
out = unet(x)
print(f"Output shape: {out.shape}")